# Track 1 — Smart Money (Financial Bot)
### EP0418 · AI for Real-World Data and Automated Decisions · Starter Pack

**Who this suits:** business or finance-curious students.
**Data:** daily OHLCV price bars (open, high, low, close, volume).
**Macro lesson:** *Baselines matter. Beating buy-and-hold is harder than it looks.*
**Your built-in baseline:** buy-and-hold the index.

> ### Rule for this track — read it before you start
> **All work is paper-trading or backtest only.** No real money, no live broker
> connection. This is enforced in the CA3 rubric, and live deployment earns
> **zero** additional marks. See the safety note in Annex A of the module syllabus.

---
This notebook loads the data, plots it, and scores the baseline you have to beat.
That's all it does — the interesting work is yours.

In [ ]:
import os, urllib.request
# Fetch this pack's data from the course repository if it is not here yet.
for _f in ['smart_money_ohlcv.csv']:
    if not os.path.exists(_f):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/starter_packs/track1_smart_money/" + _f, _f)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import os
pd.set_option("display.width", 120)

CSV = "smart_money_ohlcv.csv"          # ES3.SI — the STI ETF, ~3.5 years of daily bars
if not os.path.exists(CSV):
    from google.colab import files; files.upload()      # pick the CSV from the pack

df = pd.read_csv(CSV, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

In [ ]:
# --- OPTIONAL: pull fresh data instead of the cached CSV ---------------
# Yahoo rate-limits hard, so the pack ships cached data and this stays off.
# If it fails during a lab, that is not your bug — use the CSV above.

# !pip install -q yfinance
# import yfinance as yf
# df = yf.download("ES3.SI", start="2023-01-01", auto_adjust=True, progress=False).reset_index()
# df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
# print(df.shape)

In [ ]:
# --- FIRST LOOK -------------------------------------------------------
print("rows:", len(df), " columns:", list(df.columns))
print("date range:", df.Date.min().date(), "->", df.Date.max().date())
print("\nmissing values per column:\n", df.isna().sum())
df.head()

In [ ]:
# --- PLOT -------------------------------------------------------------
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                       gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(df.Date, df.Close, lw=1.2)
ax[0].set_title("Close price"); ax[0].set_ylabel("price"); ax[0].grid(alpha=.3)
ax[1].bar(df.Date, df.Volume, width=1.0)
ax[1].set_ylabel("volume"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# --- YOUR BASELINE: BUY AND HOLD -------------------------------------
# Buy on day 1, sell on the last day. No trading, no cleverness, no fees.
# This is the number your strategy has to beat.

df["ret"] = df.Close.pct_change()
bh_total  = df.Close.iloc[-1] / df.Close.iloc[0] - 1
years     = (df.Date.iloc[-1] - df.Date.iloc[0]).days / 365.25
bh_annual = (1 + bh_total) ** (1 / years) - 1
bh_vol    = df.ret.std() * np.sqrt(252)
bh_dd     = ((df.Close / df.Close.cummax()) - 1).min()

print(f"BUY-AND-HOLD BASELINE  ({years:.1f} years)")
print(f"  total return      {bh_total:+7.1%}")
print(f"  annualised return {bh_annual:+7.1%}")
print(f"  volatility (ann.) {bh_vol:7.1%}")
print(f"  max drawdown      {bh_dd:+7.1%}")
print("\n>>> Any strategy you build must beat these numbers AFTER costs,")
print(">>> or you must explain honestly in your report why it did not.")

---
## Three macro questions to investigate

**1. Does your strategy survive trading costs?**
A moving-average crossover that trades 40 times a year looks great until you
subtract ~0.2% per round trip. Compute your return before *and* after costs.
If the edge disappears, that is a genuine finding — report it.

**2. Are you accidentally reading the future?**
If you compute a signal from today's close and then "buy at today's close", you
have used information you would not have had. This is *lookahead bias*, and it
is the single most common way a student backtest produces a fake 300% return.
Shift your signal by one day and see how much of the edge survives.

**3. What automated response would you actually trust?**
Your CA3 must trigger something (SLO 7.3). A bot that *places orders* is out of
scope. A bot that sends you a Telegram message saying *"RSI below 30 on ES3.SI —
consider reviewing"* is in scope, and is a more honest design. What threshold
would make that alert useful rather than noisy?

## Now hand over to your AI assistant

Everything above was written for you. From here you drive. Open ChatGPT, Claude,
Copilot, or Cursor and work the **prompt → generate → verify → refine** loop
(SLO 9.1). A prompt worth copying looks like this:

> I have a pandas DataFrame `df` with columns `Date, Open, High, Low, Close, Volume, ret`.
> The time column is `Date`. Build a simple moving-average crossover backtest, shift the signal by one day to avoid lookahead bias, subtract 0.1% cost per trade, and compare total return against buy-and-hold.
> Show me the code and explain what each step does.

**Then verify before you trust it (SLO 9.2).** Every time:

| Check | How |
|---|---|
| Does it run? | Run the cell. Read the error, don't just re-prompt. |
| Right shape? | `df.shape` before and after — did you lose rows you didn't mean to? |
| Right range? | `df.describe()` — any impossible values (negative rainfall, 300 bpm)? |
| Spot-check | Pick 2 rows by hand and confirm the number is what you'd compute yourself. |
| Beat baseline? | Compare against the baseline above. If it loses, say so honestly. |

**Keep a `verification_log.md`** as you go — one line per AI mistake you caught and
how you fixed it. That log is assessed in CA3, and an empty one reads as
"I did not check", not "the AI was perfect".